In [1]:
# %% [markdown]
# # HMM Backtester Validation
# Run this in Jupyter to validate `hmm_backtester.py` against your Pine HHM chart.
#
# **Workflow:**
# 1. Pick a symbol + recent date range where you have Pine signals visible
# 2. Run the backtester with verbose tracing
# 3. Compare the single-day bar-by-bar trace against Pine's labels/arrows
# 4. Check: IB range, confluence scores, entry time/price, stop, exit
#
# Drop this notebook in `C:\Users\phemm\orb_lab\` (or adjust the path below).

# %%
import sys, os
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
from datetime import datetime
from hmm_backtester import HMMBacktester, HMM_SYMBOL_PRESETS

# %% [markdown]
# ## 1. Configuration
# Pick a symbol and a SHORT date range (1-5 days) where you can eyeball Pine.

# %%
# ═══════════════════════════════════════════════════════════════
# EDIT THESE — match your Pine chart
# ═══════════════════════════════════════════════════════════════
SYMBOL       = 'AMD'
START_DATE   = '2026-02-10'   # Pick a day you have Pine labels for
END_DATE     = '2026-02-10'   # Single day = easiest to compare

# ── Confluence settings (must match Pine exactly) ──
MIN_CONF     = 3              # Pine auto-clamps to 3 when IB is off
INCLUDE_IB   = False          # Pine: "Include IB Breakout as Signal" = UNCHECKED
USE_IB_PHASE = False          # Pine: "Use IB Phase Scoring" = OFF for this validation

# ── Quality gates (match Pine toggles) ──
USE_HTF      = True           # Pine: useHTFTrendFilter
USE_VOL_FILT = True           # Pine: useVolatilityFilter
SKIP_LUNCH   = True           # Pine: avoidLunchHour

# %%
print(f"Preset for {SYMBOL}:")
preset = HMM_SYMBOL_PRESETS.get(SYMBOL, {})
for k, v in sorted(preset.items()):
    print(f"  {k:30s} = {v}")

# %% [markdown]
# ## 2. Run Backtester (verbose=True for bar-by-bar trace)

# %%
bt = HMMBacktester(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    min_confluence=MIN_CONF,
    include_ib_in_confluence=INCLUDE_IB,
    use_ib_phase_scoring=USE_IB_PHASE,
    use_htf_trend_filter=USE_HTF,
    use_volatility_filter=USE_VOL_FILT,
    avoid_lunch_hour=SKIP_LUNCH,
    verbose=True,
)
results = bt.run()

# Print data range for sanity check
unique_dates = sorted(set(bt.df.index.date))
if unique_dates:
    print(f"\n📅 Data range: {unique_dates[0]} to {unique_dates[-1]} ({len(unique_dates)} days)")
    req_start = pd.to_datetime(START_DATE).date()
    req_end = pd.to_datetime(END_DATE).date()
    if req_start < unique_dates[0] or req_end > unique_dates[-1]:
        print(f"⚠  Requested {START_DATE} to {END_DATE} — check overlap with data range!")

# %% [markdown]
# ## 3. Trade Summary Table

# %%
if results['trades']:
    trades_df = pd.DataFrame(results['trades'])
    display(trades_df[[
        'date', 'direction', 'entry_time', 'exit_time', 'entry_price',
        'exit_price', 'exit_reason', 'stop_type', 'r_multiple',
        'confluence', 'ib_phase', 'vol_state'
    ]])
    print(f"\nTotal: {len(trades_df)} trades | "
          f"{results['total_r']:+.2f}R | "
          f"WR {results['win_rate']:.1f}% | "
          f"PF {results['profit_factor']:.2f}")
else:
    print("No trades taken. Check confluence threshold or date range.")

# %%
if results.get('skips'):
    skips_df = pd.DataFrame(results['skips'])
    print(f"\n=== SKIPPED SIGNALS ({len(skips_df)}) ===")
    display(skips_df)

# %% [markdown]
# ## 4. Single-Day Bar-by-Bar Tracer
# This is the money shot — traces every RTH bar so you can match
# against Pine's IB box, confluence table, and entry arrows.
#
# **IB columns show 0.00 when include_ib_in_confluence=False** — that's correct.
# Score = SSL + WAE + QQE + Vol only (max 4, need 3+).

# %%
def trace_day(bt, target_date: str):
    """
    Detailed bar-by-bar trace for one day.
    Prints IB build, confluence scores, quality gates, entry eval.
    Compare each line against your Pine chart bar-by-bar.
    """
    from confluence_indicators import pine_round
    
    df = bt.df
    day_df = df[df.index.date.astype(str) == target_date].copy()
    rth_df = day_df[day_df['is_rth']].copy()
    
    if len(rth_df) == 0:
        print(f"No RTH data for {target_date}")
        # Show available dates nearby
        all_dates = sorted(set(df.index.date.astype(str)))
        nearby = [d for d in all_dates if abs(pd.to_datetime(d) - pd.to_datetime(target_date)).days <= 5]
        if nearby:
            print(f"  Nearby dates with data: {', '.join(nearby[-10:])}")
        return
    
    print(f"\n{'='*90}")
    print(f"  BAR-BY-BAR TRACE: {bt.symbol} {target_date}")
    print(f"  IB Period: {bt.ib_period_minutes} min | Min Confluence: {bt.min_confluence}")
    print(f"  IB in Confluence: {bt.include_ib_in_confluence} | IB Phase Scoring: {bt.use_ib_phase_scoring}")
    print(f"  Score = SSL + WAE + QQE + Vol{' + IB' if bt.include_ib_in_confluence else ''}")
    print(f"{'='*90}\n")
    
    # Reset state for clean trace
    from hmm_backtester import IBState, VolState, Position
    ib = IBState()
    vol = VolState()
    
    ib_end_time = 9*60 + 30 + bt.ib_period_minutes
    
    if bt.include_ib_in_confluence:
        print(f"{'Time':>6}  {'O':>8} {'H':>8} {'L':>8} {'C':>8}  "
              f"{'ATR':>6} {'Vol':>5}  {'SSL':>4} {'WAE':>4} {'QQE':>4} {'VOL':>4}  "
              f"{'IB_B':>5} {'IB_S':>5}  {'Phase':>6}  Notes")
    else:
        print(f"{'Time':>6}  {'O':>8} {'H':>8} {'L':>8} {'C':>8}  "
              f"{'ATR':>6} {'Vol':>5}  {'SSL':>4} {'WAE':>4} {'QQE':>4} {'VOL':>4}  "
              f"{'Bull':>5} {'Bear':>5}  Notes")
    print("-" * 120)
    
    for i, (ts, bar) in enumerate(rth_df.iterrows()):
        time_str = ts.strftime('%H:%M')
        bar_minutes = ts.hour * 60 + ts.minute
        
        atr = bar['atr_rth']
        notes = []
        
        # ── Vol state ──
        atr_fast = bar.get('atr_fast_rth', atr)
        atr_fast_sma = bar.get('atr_fast_sma_rth', atr_fast)
        if pd.isna(atr_fast): atr_fast = atr
        if pd.isna(atr_fast_sma): atr_fast_sma = atr_fast
        
        is_orb_vol_window = 9*60+30 <= bar_minutes < 10*60
        if bar_minutes == 9*60+30:
            vol.orb_session_baseline = np.nan
        if is_orb_vol_window:
            if pd.isna(vol.orb_session_baseline):
                vol.orb_session_baseline = atr_fast_sma
            else:
                vol.orb_session_baseline = 0.9 * vol.orb_session_baseline + 0.1 * atr_fast_sma
        
        if not pd.isna(vol.orb_session_baseline) and vol.orb_session_baseline > 0:
            session_vf = atr_fast / vol.orb_session_baseline
        else:
            session_vf = 1.0
        daily_atr = bar.get('daily_atr', 1.0)
        daily_atr_slow = bar.get('daily_atr_slow', daily_atr)
        if pd.isna(daily_atr): daily_atr = 1.0
        if pd.isna(daily_atr_slow) or daily_atr_slow <= 0: daily_atr_slow = daily_atr
        htf_vf = daily_atr / daily_atr_slow if daily_atr_slow > 0 else 1.0
        vf = session_vf * htf_vf
        
        if vf < bt.low_vol_threshold:
            vs = "LOW"
        elif vf >= bt.extreme_vol_threshold:
            vs = "XTRM"
        elif vf >= bt.high_vol_threshold:
            vs = "HIGH"
        else:
            vs = "NORM"
        
        # ── IB building ──
        is_ib_window = 9*60+30 <= bar_minutes < ib_end_time
        if bar_minutes == 9*60+30:
            ib.reset_session()
            ib.ib_high = bar['high']
            ib.ib_low = bar['low']
            notes.append("IB_START")
        elif is_ib_window:
            if not pd.isna(ib.ib_high):
                ib.ib_high = max(ib.ib_high, bar['high'])
                ib.ib_low = min(ib.ib_low, bar['low'])
        
        if not is_ib_window and not ib.ib_formed and not pd.isna(ib.ib_high):
            ib.ib_formed = True
            notes.append(f"IB_FORMED H={ib.ib_high:.2f} L={ib.ib_low:.2f} R={ib.ib_high-ib.ib_low:.2f}")
        
        # ── IB phase (tracked even when not in confluence, for reference) ──
        ib_score_b = 0.0
        ib_score_s = 0.0
        phase_str = "---"
        if ib.ib_formed:
            close = bar['close']
            ib_buffer = atr * bt.ib_buffer_atr_mult if not pd.isna(atr) else 0
            ib_range = ib.ib_high - ib.ib_low
            
            if close > (ib.ib_high + ib_buffer) and not ib.ever_broke_up:
                ib.ever_broke_up = True
                notes.append("IB_BREAK_UP")
            if close < (ib.ib_low - ib_buffer) and not ib.ever_broke_down:
                ib.ever_broke_down = True
                notes.append("IB_BREAK_DN")
            
            if ib_range > 0 and ib.ever_broke_up:
                r25 = ib.ib_high - ib_range * 0.25
                r50 = ib.ib_high - ib_range * 0.50
                if bt.use_ib_phase_scoring:
                    if close > ib.ib_high:
                        ib_score_b = 1.0; phase_str = "IMP"
                    elif close >= r25:
                        ib_score_b = 1.25; phase_str = "PB25"
                    elif close >= r50:
                        ib_score_b = 0.5; phase_str = "PB50"
                    else:
                        ib_score_b = 0.0; phase_str = "FAIL"
                else:
                    # Legacy binary
                    ib_score_b = 1.0 if close > (ib.ib_high + ib_buffer) else 0.0
                    phase_str = "BRK" if ib_score_b > 0 else "---"
            
            if ib_range > 0 and ib.ever_broke_down:
                r25 = ib.ib_low + ib_range * 0.25
                r50 = ib.ib_low + ib_range * 0.50
                if bt.use_ib_phase_scoring:
                    if close < ib.ib_low:
                        ib_score_s = 1.0
                    elif close <= r25:
                        ib_score_s = 1.25
                    elif close <= r50:
                        ib_score_s = 0.5
                    else:
                        ib_score_s = 0.0
                else:
                    ib_score_s = 1.0 if close < (ib.ib_low - ib_buffer) else 0.0
        
        # ── Confluence scores ──
        full_idx = df.index.get_loc(ts)
        ssl_b = df['ssl_score_bull'].iloc[full_idx]
        ssl_s = df['ssl_score_bear'].iloc[full_idx]
        wae_b = df['wae_score_bull'].iloc[full_idx]
        wae_s = df['wae_score_bear'].iloc[full_idx]
        qqe_b = df['qqe_score_bull'].iloc[full_idx]
        qqe_s = df['qqe_score_bear'].iloc[full_idx]
        vol_sc = df['vol_score'].iloc[full_idx]
        
        # Build totals — respect IB toggle
        bull_total = ssl_b + wae_b + qqe_b + pine_round(vol_sc)
        bear_total = ssl_s + wae_s + qqe_s + pine_round(vol_sc)
        
        if bt.include_ib_in_confluence:
            bull_total += ib_score_b
            bear_total += ib_score_s
        
        # ── Quality gates ──
        htf_ema = bar.get('htf_ema', np.nan)
        htf_bull = pd.isna(htf_ema) or bar['close'] > htf_ema
        htf_bear = pd.isna(htf_ema) or bar['close'] < htf_ema
        
        atr_pct = bar.get('atr_pct', 0)
        atr_pct_avg = bar.get('atr_pct_avg', atr_pct)
        vol_ok = (not bt.use_volatility_filter or pd.isna(atr_pct_avg) or 
                  atr_pct_avg <= 0 or atr_pct >= atr_pct_avg * 0.8)
        
        is_lunch = bt.avoid_lunch_hour and (11*60+30 <= bar_minutes < 13*60+30)
        is_entry_window = (ib_end_time <= bar_minutes < 15*60+55)
        
        # ── Signal check ──
        if is_entry_window and not is_lunch and vol_ok and ib.ib_formed:
            if bull_total >= bt.min_confluence and htf_bull:
                notes.append(f"★BULL_SIG C={bull_total:.1f}")
            if bear_total >= bt.min_confluence and htf_bear:
                notes.append(f"★BEAR_SIG C={bear_total:.1f}")
        
        if is_lunch:
            notes.append("LUNCH")
        if not vol_ok:
            notes.append("VOL_SKIP")
        if is_ib_window:
            notes.append("IB_BUILD")
        
        # ── Print ──
        if bt.include_ib_in_confluence:
            print(f"{time_str:>6}  {bar['open']:>8.2f} {bar['high']:>8.2f} "
                  f"{bar['low']:>8.2f} {bar['close']:>8.2f}  "
                  f"{atr:>6.3f} {vs:>5}  "
                  f"{ssl_b:>4.0f} {wae_b:>4.0f} {qqe_b:>4.0f} {pine_round(vol_sc):>4}  "
                  f"{ib_score_b:>5.2f} {ib_score_s:>5.2f}  "
                  f"{phase_str:>6}  "
                  f"{'  '.join(notes)}")
        else:
            print(f"{time_str:>6}  {bar['open']:>8.2f} {bar['high']:>8.2f} "
                  f"{bar['low']:>8.2f} {bar['close']:>8.2f}  "
                  f"{atr:>6.3f} {vs:>5}  "
                  f"{ssl_b:>4.0f} {wae_b:>4.0f} {qqe_b:>4.0f} {pine_round(vol_sc):>4}  "
                  f"{bull_total:>5.1f} {bear_total:>5.1f}  "
                  f"{'  '.join(notes)}")
    
    # Summary
    print(f"\n{'─'*90}")
    print(f"  IB Range: {ib.ib_high:.2f} - {ib.ib_low:.2f} = {ib.ib_high - ib.ib_low:.2f}")
    print(f"  Broke Up: {ib.ever_broke_up}  Broke Down: {ib.ever_broke_down}")
    print(f"  Score formula: SSL + WAE + QQE + Vol{' + IB' if bt.include_ib_in_confluence else ''} (max {'5' if bt.include_ib_in_confluence else '4'}, need {bt.min_confluence}+)")
    print(f"{'─'*90}")

# %%
# Run the tracer on your target date
trace_day(bt, START_DATE)

# %% [markdown]
# ## 5. Pine Comparison Checklist
#
# Open your Pine HHM chart on the same symbol/date and verify:
#
# | Check | Python | Pine | Match? |
# |-------|--------|------|--------|
# | IB High | from trace | IB box top | |
# | IB Low | from trace | IB box bottom | |
# | IB Break direction | from trace | IB break label | |
# | First signal time | ★BULL/BEAR_SIG | entry arrow | |
# | Entry price | trades_df | entry label | |
# | Stop price | trades_df | stop line | |
# | Confluence score | trades_df | confluence table | |
# | Exit time/reason | trades_df | exit label | |
# | SSL/WAE/QQE per bar | trace columns | indicator panel | |
#
# **Common mismatches to watch for:**
# - IB period: Python uses `ib_period_minutes` from preset, Pine uses `ibPeriodMinutesEff`
# - HTF EMA: Python resamples 1-min→5-min, Pine uses `request.security("5",...)`
# - WAE explosion: Python computes RTH-only BB, Pine uses `request.security`
# - Lunch skip: 11:30-13:30 ET in both — verify timezone alignment

# %% [markdown]
# ## 6. Multi-Day Quick Scan
# Run a wider window and inspect the aggregate to see if trade count/direction
# roughly matches what Pine shows in Strategy Tester.

# %%
bt_wide = HMMBacktester(
    symbol=SYMBOL,
    start_date='2026-01-06',    # ← Adjust to your Pine visible range
    end_date='2026-02-10',
    min_confluence=MIN_CONF,
    include_ib_in_confluence=INCLUDE_IB,
    use_ib_phase_scoring=USE_IB_PHASE,
    use_htf_trend_filter=USE_HTF,
    use_volatility_filter=USE_VOL_FILT,
    avoid_lunch_hour=SKIP_LUNCH,
    verbose=False,
)
results_wide = bt_wide.run()

print(f"\n{'='*70}")
print(f"  MULTI-DAY SCAN: {SYMBOL}")
print(f"  {results_wide['start_date']} to {results_wide['end_date']}")
print(f"{'='*70}")
print(f"  Trades: {results_wide['total_trades']}  "
      f"W/L: {results_wide['wins']}W/{results_wide['losses']}L  "
      f"WR: {results_wide['win_rate']:.1f}%")
print(f"  Total R: {results_wide['total_r']:+.2f}  "
      f"Avg R: {results_wide['avg_r_per_trade']:+.3f}  "
      f"PF: {results_wide['profit_factor']:.2f}")

if results_wide['trades']:
    wide_df = pd.DataFrame(results_wide['trades'])
    print(f"\n  Direction split:")
    for d in ['LONG', 'SHORT']:
        sub = wide_df[wide_df['direction'] == d]
        if len(sub) > 0:
            wr = 100 * len(sub[sub['r_multiple'] > 0]) / len(sub)
            print(f"    {d:5s}: {len(sub)} trades | WR {wr:.0f}% | "
                  f"R {sub['r_multiple'].sum():+.2f}")
    
    print(f"\n  IB Phase distribution:")
    print(wide_df['ib_phase'].value_counts().to_string())
    
    print(f"\n  Exit reason distribution:")
    print(wide_df['exit_reason'].value_counts().to_string())
    
    print(f"\n  Vol state distribution:")
    print(wide_df['vol_state'].value_counts().to_string())
    
    # Per-day summary
    print(f"\n  {'Date':>12} {'Dir':>6} {'Entry':>6} {'Exit':>6} "
          f"{'R':>7} {'Conf':>5} {'IB':>5} {'Reason'}")
    print(f"  {'-'*70}")
    for _, t in wide_df.iterrows():
        print(f"  {t['date']:>12} {t['direction']:>6} {t['entry_time']:>6} "
              f"{t['exit_time']:>6} {t['r_multiple']:>+7.2f} "
              f"{t['confluence']:>5.1f} {t['ib_phase']:>5} {t['exit_reason']}")

# %% [markdown]
# ## 7. Indicator Snapshot (Debug Helper)
# If a specific bar's confluence doesn't match Pine, dump the raw indicator
# values to find which one diverges.

# %%
def indicator_snapshot(bt, target_date: str, target_time: str):
    """Dump all indicator values at a specific bar for debugging."""
    df = bt.df
    target_ts = pd.Timestamp(f"{target_date} {target_time}", tz='America/New_York')
    
    # Find closest bar
    idx = df.index.get_indexer([target_ts], method='nearest')[0]
    bar = df.iloc[idx]
    ts = df.index[idx]
    
    print(f"\n{'='*60}")
    print(f"  INDICATOR SNAPSHOT: {bt.symbol} {ts}")
    print(f"{'='*60}")
    
    print(f"\n  OHLCV:")
    print(f"    Open:   {bar['open']:.4f}")
    print(f"    High:   {bar['high']:.4f}")
    print(f"    Low:    {bar['low']:.4f}")
    print(f"    Close:  {bar['close']:.4f}")
    print(f"    Volume: {bar['volume']:.0f}")
    
    print(f"\n  ATR / VWAP / EMA:")
    print(f"    ATR (RTH):     {bar.get('atr_rth', 'N/A')}")
    print(f"    ATR (ETH):     {bar.get('atr_eth', 'N/A')}")
    print(f"    VWAP:          {bar.get('vwap', 'N/A')}")
    print(f"    EMA9:          {bar.get('ema9', 'N/A')}")
    print(f"    HTF EMA (5m):  {bar.get('htf_ema', 'N/A')}")
    
    print(f"\n  SSL Hybrid:")
    print(f"    Baseline:      {bar.get('ssl_baseline', 'N/A')}")
    print(f"    SSL Up:        {bar.get('ssl_up', 'N/A')}")
    print(f"    SSL Down:      {bar.get('ssl_down', 'N/A')}")
    print(f"    Gap bull:      {bar.get('ssl_gap_bull', 'N/A')}")
    print(f"    Gap bear:      {bar.get('ssl_gap_bear', 'N/A')}")
    print(f"    Bull confirm:  {bar.get('ssl_bull_confirm', 'N/A')}")
    print(f"    Bear confirm:  {bar.get('ssl_bear_confirm', 'N/A')}")
    print(f"    Accel bull:    {bar.get('ssl_accel_bull', 'N/A')}")
    print(f"    Accel bear:    {bar.get('ssl_accel_bear', 'N/A')}")
    print(f"    Score bull:    {bar.get('ssl_score_bull', 'N/A')}")
    print(f"    Score bear:    {bar.get('ssl_score_bear', 'N/A')}")
    
    print(f"\n  WAE:")
    print(f"    Trend up:      {bar.get('wae_trend_up', 'N/A')}")
    print(f"    Trend down:    {bar.get('wae_trend_down', 'N/A')}")
    print(f"    Explosion:     {bar.get('wae_explosion_line', 'N/A')}")
    print(f"    Bull confirm:  {bar.get('wae_bull_confirm', 'N/A')}")
    print(f"    Bear confirm:  {bar.get('wae_bear_confirm', 'N/A')}")
    print(f"    Accel bull:    {bar.get('wae_accel_bull', 'N/A')}")
    print(f"    Accel bear:    {bar.get('wae_accel_bear', 'N/A')}")
    print(f"    Score bull:    {bar.get('wae_score_bull', 'N/A')}")
    print(f"    Score bear:    {bar.get('wae_score_bear', 'N/A')}")
    
    print(f"\n  QQE:")
    print(f"    Primary RSI:   {bar.get('qqe_primary_rsi', 'N/A')}")
    print(f"    Secondary RSI: {bar.get('qqe_secondary_rsi', 'N/A')}")
    print(f"    Primary QQE:   {bar.get('qqe_primary_line', 'N/A')}")
    print(f"    BB upper:      {bar.get('qqe_bb_upper', 'N/A')}")
    print(f"    BB lower:      {bar.get('qqe_bb_lower', 'N/A')}")
    print(f"    Raw bull:      {bar.get('qqe_raw_bull', 'N/A')}")
    print(f"    Raw bear:      {bar.get('qqe_raw_bear', 'N/A')}")
    print(f"    Bull count:    {bar.get('qqe_bull_count', 'N/A')}")
    print(f"    Bear count:    {bar.get('qqe_bear_count', 'N/A')}")
    print(f"    Bull signal:   {bar.get('qqe_bull_signal', 'N/A')}")
    print(f"    Bear signal:   {bar.get('qqe_bear_signal', 'N/A')}")
    print(f"    Momentum rise: {bar.get('qqe_momentum_rising', 'N/A')}")
    print(f"    Momentum fall: {bar.get('qqe_momentum_falling', 'N/A')}")
    print(f"    Score bull:    {bar.get('qqe_score_bull', 'N/A')}")
    print(f"    Score bear:    {bar.get('qqe_score_bear', 'N/A')}")
    
    print(f"\n  Volume:")
    print(f"    Volume:        {bar.get('volume', 'N/A')}")
    print(f"    Vol Z-score:   {bar.get('vol_z', 'N/A')}")
    print(f"    Vol score:     {bar.get('vol_score', 'N/A')}")
    print(f"    Vol rounded:   {pine_round(bar.get('vol_score', 0))}")
    
    print(f"\n  Volatility:")
    print(f"    ATR %:         {bar.get('atr_pct', 'N/A')}")
    print(f"    ATR % avg:     {bar.get('atr_pct_avg', 'N/A')}")
    print(f"    Daily ATR:     {bar.get('daily_atr', 'N/A')}")
    print(f"    Daily ATR slow:{bar.get('daily_atr_slow', 'N/A')}")
    
    # Import pine_round for total calc
    from confluence_indicators import pine_round
    ssl_b = bar.get('ssl_score_bull', 0)
    wae_b = bar.get('wae_score_bull', 0)
    qqe_b = bar.get('qqe_score_bull', 0)
    vol_s = bar.get('vol_score', 0)
    ssl_s = bar.get('ssl_score_bear', 0)
    wae_s = bar.get('wae_score_bear', 0)
    qqe_s = bar.get('qqe_score_bear', 0)
    
    bull_t = ssl_b + wae_b + qqe_b + pine_round(vol_s)
    bear_t = ssl_s + wae_s + qqe_s + pine_round(vol_s)
    
    print(f"\n  ── CONFLUENCE TOTALS (IB excluded) ──")
    print(f"    Bull: SSL({ssl_b:.0f}) + WAE({wae_b:.0f}) + QQE({qqe_b:.0f}) + Vol({pine_round(vol_s)}) = {bull_t:.0f}  {'✓ PASS' if bull_t >= bt.min_confluence else '✗ FAIL'}")
    print(f"    Bear: SSL({ssl_s:.0f}) + WAE({wae_s:.0f}) + QQE({qqe_s:.0f}) + Vol({pine_round(vol_s)}) = {bear_t:.0f}  {'✓ PASS' if bear_t >= bt.min_confluence else '✗ FAIL'}")

# Example usage — change date/time to the bar you want to inspect:
# indicator_snapshot(bt, '2026-02-10', '10:35')

Preset for AMD:
  extreme_vol_threshold          = 2.0
  high_vol_threshold             = 1.3
  ib_period_minutes              = 52
  low_vol_threshold              = 0.8
  qqe_bb_length                  = 25
  qqe_rsi1_length                = 8
  qqe_rsi1_smoothing             = 5
  qqe_rsi2_length                = 4
  qqe_rsi2_smoothing             = 4
  ssl_baseline_length            = 60
  ssl_length                     = 10
  vol_lookback                   = 12
  wae_bb_length                  = 20
  wae_fast_ema                   = 14
  wae_sensitivity                = 300
  wae_slow_ema                   = 29
[HMM Presets] AMD: SSL_base=60, WAE_sens=300, QQE_rsi1=8, IB_min=52

  HMM BACKTESTER: AMD
  2026-02-10 to 2026-02-10
  Min Confluence: 3  IB Phase: False

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (0.3 hours old)
  Loaded 101126 1-min bars for AMD
  Resampled to 54943 2-min bars


,date,direction,entry_time,exit_time,entry_price,exit_price,exit_reason,stop_type,r_multiple,confluence,ib_phase,vol_state
0,2026-02-10,LONG,10:22,11:08,216.2100,216.670000,EMA EXIT,SWING,0.346345,3.0,---,LOW
1,2026-02-10,LONG,11:10,11:22,217.3101,216.548475,INITIAL STOP,SWING,-1.000000,3.0,---,LOW
2,2026-02-10,LONG,13:58,14:18,215.4250,214.924337,INITIAL STOP,SWING,-1.000000,3.0,---,LOW
3,2026-02-10,SHORT,14:18,14:30,215.0000,215.406114,INITIAL STOP,SWING,-1.000000,3.0,---,LOW



Total: 4 trades | -2.65R | WR 25.0% | PF 0.28

=== SKIPPED SIGNALS (12) ===


,date,time,direction,entry_price,achievable_rr,min_rr,reason
0,2026-02-10,13:34,SHORT,214.390,1.500000,2.0,RR 1.50 < 2.0
1,2026-02-10,14:30,LONG,215.600,1.500000,2.0,RR 1.50 < 2.0
2,2026-02-10,14:44,SHORT,215.060,1.692167,2.0,RR 1.69 < 2.0
3,2026-02-10,14:46,SHORT,214.900,1.500000,2.0,RR 1.50 < 2.0
4,2026-02-10,14:50,SHORT,214.800,1.500000,2.0,RR 1.50 < 2.0
5,2026-02-10,14:56,LONG,215.310,1.503806,2.0,RR 1.50 < 2.0
6,2026-02-10,15:00,SHORT,214.364,1.500000,2.0,RR 1.50 < 2.0
7,2026-02-10,15:08,SHORT,214.320,1.500000,2.0,RR 1.50 < 2.0
8,2026-02-10,15:10,SHORT,214.080,1.500000,2.0,RR 1.50 < 2.0
9,2026-02-10,15:12,SHORT,214.000,1.500000,2.0,RR 1.50 < 2.0



  BAR-BY-BAR TRACE: AMD 2026-02-10
  IB Period: 52 min | Min Confluence: 3
  IB in Confluence: False | IB Phase Scoring: False
  Score = SSL + WAE + QQE + Vol

  Time         O        H        L        C     ATR   Vol   SSL  WAE  QQE  VOL   Bull  Bear  Notes
------------------------------------------------------------------------------------------------------------------------
 09:30    215.14   219.18   215.10   217.87   0.634  XTRM     1    1    0    1    3.0   1.0  IB_START  IB_BUILD
 09:32    217.85   218.72   216.45   216.97   0.751  XTRM     0    0    0    1    1.0   2.0  IB_BUILD
 09:34    217.11   219.36   216.26   219.36   0.919  XTRM     1    1    0    1    3.0   1.0  IB_BUILD
 09:36    219.35   219.39   216.67   216.81   1.047  XTRM     0    0    0    1    1.0   3.0  IB_BUILD
 09:38    216.85   217.68   216.42   216.57   1.063  XTRM     0    0    0    1    1.0   3.0  IB_BUILD
 09:40    216.48   216.94   215.41   216.56   1.096  XTRM     0    0    0    1    1.0   2.0  IB_BUI

In [1]:
bt = HMMBacktester(
    symbol='AMD',
    start_date='2026-01-10',
    end_date='2026-02-10',
    bar_size=2,
    verbose=True
)
bt.load_data()
bt.run()
results = bt.run()

NameError: name 'HMMBacktester' is not defined

In [2]:
from hmm_backtester import HMMBacktester

bt = HMMBacktester(
    symbol='AMD',
    start_date='2026-01-10',
    end_date='2026-02-10',
    bar_size=2,
    verbose=True
)
bt.load_data()
results = bt.run()

ModuleNotFoundError: No module named 'hmm_backtester'

In [3]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

from hmm_backtester import HMMBacktester

bt = HMMBacktester(
    symbol='AMD',
    start_date='2026-01-10',
    end_date='2026-02-10',
    bar_size=2,
    verbose=True
)
bt.load_data()
results = bt.run()

[HMM Presets] AMD: SSL_base=60, WAE_sens=300, QQE_rsi1=8, IB_min=52
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.9 hours old)
  Loaded 101126 1-min bars for AMD
  Resampled to 54943 2-min bars
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [OK] Confluence indicators calculated

  HMM BACKTESTER: AMD
  2026-01-10 to 2026-02-10
  Min Confluence: 3  IB Phase: True

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (1.9 hours old)
  Loaded 101126 1-min bars for AMD
  Resampled to 54943 2-min bars
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [OK] Confluence indicators calculated
  Data range: 2025-08-18 to 2026-02-12 (124 days)
Processing 21 trading days...


--- 2026-01-12 ---
  [IB] Formed: H=209.17 L=199.80 Range=9.37
  [ENTRY] SHORT 10:30 @ $207.85 stop=$209.16 (

In [4]:
# After bt.run(), dump Feb 10 confluence scores
df = bt.df
feb10 = df[df.index.date == pd.Timestamp('2026-02-10').date()]
rth = feb10[(feb10.index.hour*60 + feb10.index.minute >= 570) & 
            (feb10.index.hour*60 + feb10.index.minute < 960)]

cols = ['open','high','low','close','ssl_score_bull','ssl_score_bear',
        'wae_score_bull','wae_score_bear','qqe_score_bull','qqe_score_bear',
        'vol_score']
print(rth[cols].to_string())

NameError: name 'pd' is not defined

In [1]:
import pandas as pd

df = bt.df
feb10 = df[df.index.date == pd.Timestamp('2026-02-10').date()]
rth = feb10[(feb10.index.hour*60 + feb10.index.minute >= 570) & 
            (feb10.index.hour*60 + feb10.index.minute < 960)]

cols = ['open','high','low','close','ssl_score_bull','ssl_score_bear',
        'wae_score_bull','wae_score_bear','qqe_score_bull','qqe_score_bear',
        'vol_score']
print(rth[cols].to_string())

NameError: name 'bt' is not defined

In [2]:
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import importlib
import hmm_backtester, confluence_indicators
importlib.reload(confluence_indicators)
importlib.reload(hmm_backtester)
from hmm_backtester import HMMBacktester
import pandas as pd

bt = HMMBacktester(
    symbol='AMD',
    start_date='2026-01-10',
    end_date='2026-02-10',
    bar_size=2,
    verbose=True
)
bt.load_data()
results = bt.run()

[HMM Presets] AMD: SSL_base=60, WAE_sens=300, QQE_rsi1=8, IB_min=52
[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.1 hours old)
  Loaded 101126 1-min bars (ETH+RTH) for AMD
  Filtered to 48002 RTH bars
  Resampled to 24002 2-min bars
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [OK] Confluence indicators calculated

  HMM BACKTESTER: AMD
  2026-01-10 to 2026-02-10
  Min Confluence: 3  IB Phase: True

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching AMD (1-min bars, 180 days, extended)...
  ✓ Loading from cache (4.1 hours old)
  Loaded 101126 1-min bars (ETH+RTH) for AMD
  Filtered to 48002 RTH bars
  Resampled to 24002 2-min bars
  [*] Calculating confluence indicators (SSL, WAE, QQE, Volume)...
  [OK] Confluence indicators calculated
  Data range: 2025-08-18 to 2026-02-12 (124 days)
Processing 21 trading days...


--- 2026-01-12 ---
  [IB] Forme

In [4]:
df = bt.df
bar = df.loc['2026-02-10 14:18:00-05:00']
print(f"SSL bull={bar['ssl_score_bull']} bear={bar['ssl_score_bear']}")
print(f"WAE bull={bar['wae_score_bull']} bear={bar['wae_score_bear']}")
print(f"QQE bull={bar['qqe_score_bull']} bear={bar['qqe_score_bear']}")
print(f"Vol={bar['vol_score']}")
print(f"Close={bar['close']}")

SSL bull=0.0 bear=1.0
WAE bull=0.0 bear=1.0
QQE bull=0.0 bear=0.0
Vol=0.5
Close=215.0
